# 🧠 Taller: Redes Neuronales desde Cero
## Construyendo una neurona sigmoidal con NumPy, paso a paso

**Curso:** Inteligencia Artificial con Aplicaciones en Economía II
**Insumos:** el notebook de clase `Red_Neuronal_desde_Cero_NumPy_Economistas.ipynb` y la lectura *"Guías de IA para Econ – Neural Networks"*
**Herramientas:** únicamente `numpy`, `pandas` y `matplotlib` — nada de `scikit-learn`, `tensorflow` ni `pytorch`.

---

## Objetivo del taller

Este taller acompaña la clase de redes neuronales y tiene dos metas:

1. Repasar y aplicar los **conceptos básicos** de una neurona artificial (perceptrón, sigmoide, función de costo, descenso del gradiente).
2. **Construir con sus propias manos** una red neuronal mínima (una neurona sigmoidal entrenable) que clasifica riesgo económico, y experimentar con ella.

No es un ejercicio de copiar y pegar: varias celdas están incompletas a propósito. Su trabajo es completarlas usando lo visto en clase y en la lectura.

---

## Cómo usar este taller

- Busque los comentarios `# TODO` — ahí debe escribir código. Todo lo demás ya está resuelto para que pueda concentrarse en las partes conceptualmente importantes.
- Las celdas de texto con **"✍️ Tu respuesta:"** son para responder preguntas conceptuales con sus propias palabras. No hay una única respuesta "correcta" — lo importante es el razonamiento.
- Ejecute las celdas **en orden**: cada una depende de las anteriores (en particular, reutilizamos varias veces las funciones `sigmoid`, `mse_cost`, etc. que usted mismo va a implementar).
- Si se atasca, puede volver al notebook de la clase o a la guía de la clase — la lógica es la misma, solo cambia que aquí usted la escribe.

## Estructura y duración sugerida

| Parte | Contenido | Sesión sugerida |
|---|---|---|
| **Parte 1** | Fundamentos: perceptrón, sigmoide, función de costo | Sesión 1 |
| **Parte 2** | Construcción de la clase `Network` y su entrenamiento | Sesión 1 (cierre) |
| **Parte 3** | Experimentos: tasa de aprendizaje, inicialización, umbral de decisión | Sesión 2 |

Duración total estimada: 90–120 minutos, repartidos en las dos sesiones de la clase.


---
# Sección 0 — Preparación (dado)

Esta celda no requiere cambios: solo importa las librerías y fija la semilla aleatoria para que los resultados sean reproducibles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)

---
# Parte 1 — Fundamentos de una neurona

## Ejercicio 0 — El problema económico y los datos (dado)

Simulamos un problema de decisión binaria: una institución quiere clasificar si una unidad económica tiene **alto riesgo** (`1`) o **bajo riesgo** (`0`) usando dos variables:

- \(x_1\): ingreso relativo normalizado;
- \(x_2\): nivel de deuda relativo normalizado.

Intuición económica: mayor ingreso reduce el riesgo, mayor deuda lo aumenta.

In [ ]:
def generate_economic_data(n=160, seed=42):
    """Genera un dataset sintetico con dos variables economicas.

    Args:
        n: Numero de observaciones.
        seed: Semilla para reproducibilidad.

    Returns:
        X: Matriz de predictores con columnas ingreso y deuda.
        y: Vector columna con clase binaria de riesgo.
    """
    rng = np.random.default_rng(seed)

    # Grupo 0: bajo riesgo, mayor ingreso y menor deuda
    low_risk = rng.normal(loc=[1.6, 0.8], scale=[0.45, 0.35], size=(n // 2, 2))

    # Grupo 1: alto riesgo, menor ingreso y mayor deuda
    high_risk = rng.normal(loc=[0.8, 1.7], scale=[0.40, 0.45], size=(n // 2, 2))

    X = np.vstack([low_risk, high_risk])
    y = np.vstack([np.zeros((n // 2, 1)), np.ones((n // 2, 1))])

    idx = rng.permutation(n)
    return X[idx], y[idx]


def plot_data(X, y, title="Datos sinteticos: riesgo economico"):
    """Grafica los puntos del dataset sintetico."""
    plt.figure()
    plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], label="Bajo riesgo (0)", alpha=0.8, color="#8FAE8F")
    plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], label="Alto riesgo (1)", alpha=0.8, color="#A8672B")
    plt.xlabel("Ingreso relativo normalizado")
    plt.ylabel("Deuda relativa normalizada")
    plt.title(title)
    plt.legend()
    plt.show()


def plot_decision_boundary(X, y, weights, bias, title="Frontera de decision"):
    """Grafica datos y frontera de decision para dos variables."""
    plt.figure()
    plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], label="Bajo riesgo (0)", alpha=0.8, color="#8FAE8F")
    plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], label="Alto riesgo (1)", alpha=0.8, color="#A8672B")

    x1_values = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
    if abs(weights[1]) > 1e-8:
        x2_values = -(weights[0] * x1_values + bias) / weights[1]
        plt.plot(x1_values, x2_values, label="Frontera", linewidth=2, color="#2F5233")

    plt.xlabel("Ingreso relativo normalizado")
    plt.ylabel("Deuda relativa normalizada")
    plt.title(title)
    plt.legend()
    plt.show()


X, y = generate_economic_data()
print("Dimension de X:", X.shape)
print("Dimension de y:", y.shape)
print()
print("Observaciones por clase:")
print(pd.Series(y.ravel()).value_counts())

plot_data(X, y)

**✍️ Tu respuesta:** mirando el gráfico anterior, ¿por qué cree que estos datos podrían separarse razonablemente bien con una línea recta? ¿En qué zona del gráfico esperaría que el modelo cometiera más errores?

*(escriba aquí)*

## Ejercicio 1 — El perceptrón: función escalón

Un perceptrón calcula un índice lineal:

\[
z = w_1x_1 + w_2x_2 + b
\]

y aplica una regla de decisión **dura**:

\[
\hat{y} =
\begin{cases}
1, & z \geq 0\\
0, & z < 0
\end{cases}
\]

Esto se parece a una regla institucional con umbral: *"si el índice de riesgo supera cierto umbral, clasificamos como alto riesgo."*

- \(w_1\): importancia relativa del ingreso;
- \(w_2\): importancia relativa de la deuda;
- \(b\): umbral o costo de entrada.

**Complete las dos funciones de abajo.**

In [ ]:
def step_function(z):
    """Funcion escalon para el perceptron.
    Debe devolver 1 donde z >= 0, y 0 en caso contrario (use np.where).
    """
    # TODO: complete esta funcion (1 linea)
    pass


def perceptron_predict(X, weights, bias):
    """Calcula z = X @ weights + bias y le aplica step_function."""
    # TODO: complete esta funcion (2 lineas: calcular z, y devolver step_function(z))
    pass


weights_manual = np.array([-1.0, 1.0])
bias_manual = 0.0

manual_pred = perceptron_predict(X, weights_manual, bias_manual)
manual_pred[:10].ravel()

In [ ]:
plot_decision_boundary(X, y, weights_manual, bias_manual, title="Perceptrón con pesos manuales")

## Ejercicio 1b — Ajuste manual de pesos

Cambie manualmente `weights_try` y `bias_try` abajo y vuelva a ejecutar la celda.

- \(w_1\): peso del ingreso;
- \(w_2\): peso de la deuda;
- \(b\): umbral.

**Pregunta:** ¿qué ocurre con la frontera si el peso de la deuda es mucho mayor que el peso del ingreso (por ejemplo `w = [-0.3, 2.5]`)?

In [ ]:
# EJERCICIO DEL ESTUDIANTE
# Cambie estos valores y vuelva a ejecutar la celda para ver como se mueve la frontera.

weights_try = np.array([-1.0, 1.5])  # TODO: pruebe otros valores
bias_try = -0.2                       # TODO: pruebe otros valores

plot_decision_boundary(X, y, weights_try, bias_try, title="Frontera con pesos modificados por el estudiante")

**✍️ Tu respuesta:**

*(escriba aquí)*

## Ejercicio 2 — De perceptrón a neurona sigmoidal

El perceptrón cambia bruscamente de 0 a 1: eso dificulta el aprendizaje gradual, porque no se puede derivar en el punto de quiebre. Si queremos ajustar pesos poco a poco mediante optimización, necesitamos una función **suave**.

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

Esta función transforma cualquier valor real en un número entre 0 y 1 — se parece a una probabilidad estimada \(P(y=1|X)\). Su derivada tiene una forma muy simple cuando se expresa en función de su propia salida \(a = \sigma(z)\):

\[
\sigma'(z) = a(1-a)
\]

**Complete las dos funciones de abajo.**

In [ ]:
def sigmoid(z):
    # TODO: complete esta funcion (1 linea): 1 / (1 + np.exp(-z))
    pass


def sigmoid_derivative_from_output(a):
    # TODO: complete esta funcion (1 linea): a * (1 - a)
    pass


z_values = np.linspace(-8, 8, 300)

plt.figure()
plt.plot(z_values, step_function(z_values), label="Escalón / Perceptrón")
plt.plot(z_values, sigmoid(z_values), label="Sigmoide")
plt.title("Escalón vs Sigmoide")
plt.xlabel("z")
plt.ylabel("Salida")
plt.legend()
plt.show()

**✍️ Tu respuesta:** en vez de decir *"esta empresa es riesgosa o no"*, la sigmoide nos permite decir *"esta empresa tiene 0.78 de probabilidad estimada de alto riesgo"*. ¿Por qué esa diferencia es útil para una institución financiera que debe decidir cuánto crédito ofrecer o qué tasa de interés cobrar?

*(escriba aquí)*

## Ejercicio 3 — Función de costo: medir qué tan mal predice el modelo

Para que una red aprenda, necesita una medida de error. Usaremos el error cuadrático medio (MSE):

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
\]

Aunque en clasificación suele usarse entropía cruzada, el MSE es pedagógicamente útil para ver la lógica de minimización con claridad.

**Complete la función y calcule el costo inicial con los pesos manuales del Ejercicio 1.**

In [ ]:
def mse_cost(y_true, y_pred):
    # TODO: complete esta funcion (1 linea): np.mean((y_true - y_pred) ** 2)
    pass


z_manual = X @ weights_manual.reshape(-1, 1) + bias_manual
prob_manual = sigmoid(z_manual)

# TODO: calcule el costo inicial usando mse_cost(y, prob_manual)
initial_cost = None

print("Costo inicial con pesos manuales:", initial_cost)

**✍️ Tu respuesta:** en términos económicos, ¿qué representa este número? Piense en la función de costo como una función de pérdida social: ¿qué significaría que el costo fuera 0? ¿Y que fuera muy alto (cerca de 1)?

*(escriba aquí)*

## Ejercicio 4 — Fronteras manuales (dado, exploratorio)

Antes de automatizar el aprendizaje, observemos manualmente cómo cambia la frontera de decisión al cambiar los pesos.

In [ ]:
manual_configs = [
    {"weights": np.array([-0.5, 0.5]), "bias": 0.0, "name": "Frontera 1: pesos débiles"},
    {"weights": np.array([-1.0, 1.0]), "bias": 0.0, "name": "Frontera 2: pesos balanceados"},
    {"weights": np.array([-1.4, 1.8]), "bias": -0.3, "name": "Frontera 3: más peso a deuda"}
]

for config in manual_configs:
    plot_decision_boundary(X, y, config["weights"], config["bias"], title=config["name"])

**✍️ Tu respuesta:** ¿cuál de las tres fronteras le parece que separa mejor los dos grupos? Desde una perspectiva económica, ¿debería pesar más el ingreso o la deuda? ¿Qué representa mover el sesgo \(b\)?

*(escriba aquí)*

---
# Parte 2 — Construcción de la red neuronal

## Ejercicio 5 — La clase `Network` (el corazón del taller)

Ahora automatizamos el aprendizaje. Construiremos una clase `Network` que representa una neurona sigmoidal entrenable, con:

- pesos \(w\) y sesgo \(b\);
- función sigmoide (`forward`);
- función de costo (`cost`);
- entrenamiento por descenso del gradiente (`train`).

**Regla de actualización.** Queremos minimizar \(C = \frac{1}{n}\sum(y-\hat{y})^2\) usando descenso del gradiente:

\[
w \leftarrow w - \eta \frac{\partial C}{\partial w}, \qquad b \leftarrow b - \eta \frac{\partial C}{\partial b}
\]

donde \(\eta\) es la tasa de aprendizaje. Para llegar a esas derivadas usamos la **regla de la cadena** (backpropagation, aplicada a una sola neurona):

1. `y_hat = forward(X)` — la predicción actual;
2. `error = y_hat - y` — qué tan lejos está la predicción del valor real;
3. `dcost_dyhat = 2 * error / n` — derivada del MSE respecto a \(\hat{y}\);
4. `dyhat_dz = sigmoid_derivative(y_hat)` — derivada de la sigmoide (recuerde: \(a(1-a)\));
5. `delta = dcost_dyhat * dyhat_dz` — la regla de la cadena combina los dos pasos anteriores: cuánto cambia el costo por cada cambio en \(z\);
6. `dweights = X.T @ delta` — el gradiente respecto a los pesos;
7. `dbias = np.sum(delta)` — el gradiente respecto al sesgo;
8. actualizar `weights` y `bias` restando `learning_rate` veces el gradiente.

**Complete el cuerpo de la clase.** Puede (y debe) reutilizar las funciones `sigmoid`, `sigmoid_derivative_from_output` y `mse_cost` que ya implementó en la Parte 1.

In [ ]:
class Network:
    """Red neuronal minima con una neurona sigmoidal."""

    def __init__(self, n_inputs, learning_rate=0.1, seed=42, zero_init=False):
        rng = np.random.default_rng(seed)
        if zero_init:
            self.weights = np.zeros((n_inputs, 1))
        else:
            self.weights = rng.normal(loc=0.0, scale=0.1, size=(n_inputs, 1))
        self.bias = 0.0
        self.learning_rate = learning_rate
        self.history = []
        self.weight_history = []

    def sigmoid(self, z):
        # TODO: reutilice su funcion sigmoid de la Parte 1 (1 linea)
        pass

    def sigmoid_derivative(self, a):
        # TODO: reutilice su funcion sigmoid_derivative_from_output (1 linea)
        pass

    def forward(self, X):
        """Calcula z = Xw + b y le aplica la sigmoide."""
        # TODO: complete el forward pass (2 lineas: calcular z, y devolver self.sigmoid(z))
        pass

    def cost(self, y_true, y_pred):
        # TODO: reutilice la logica de mse_cost (1 linea)
        pass

    def train(self, X, y, epochs=1000, store_every=50):
        """Entrena la red usando descenso del gradiente."""
        n = X.shape[0]

        for epoch in range(epochs):
            # 1) Forward pass: prediccion actual
            y_hat = None  # TODO: self.forward(X)

            # 2) Error de prediccion
            error = None  # TODO: y_hat - y

            # 3) Derivada del MSE respecto a y_hat: 2 * error / n
            dcost_dyhat = None  # TODO

            # 4) Derivada de la sigmoide evaluada en y_hat
            dyhat_dz = None  # TODO: self.sigmoid_derivative(y_hat)

            # 5) Regla de la cadena: delta = dcost_dyhat * dyhat_dz
            delta = None  # TODO

            # 6) Gradiente respecto a los pesos: X.T @ delta
            dweights = None  # TODO

            # 7) Gradiente respecto al sesgo: suma de delta
            dbias = None  # TODO

            # 8) Actualizacion de pesos y sesgo (ya dada)
            self.weights -= self.learning_rate * dweights
            self.bias -= self.learning_rate * dbias

            if epoch % store_every == 0 or epoch == epochs - 1:
                current_cost = self.cost(y, self.forward(X))
                self.history.append({"epoch": epoch, "cost": current_cost})
                self.weight_history.append((epoch, self.weights.copy(), self.bias))

    def predict_proba(self, X):
        # TODO: devuelva self.forward(X)
        pass

    def predict(self, X, threshold=0.5):
        # TODO: convierta probabilidades en clases 0/1 usando el umbral
        # (pista: (self.predict_proba(X) >= threshold).astype(int))
        pass

## Ejercicio 6 — Entrenamiento

Instancie la red y entrénela.

In [ ]:
network = Network(n_inputs=2, learning_rate=0.5, seed=42)
network.train(X, y, epochs=2000, store_every=20)

# TODO: construya un DataFrame de pandas a partir de network.history
history_df = None

print(history_df.head())
print(history_df.tail())

## Ejercicio 7 — Visualizar la curva de costo

Grafique `history_df["epoch"]` contra `history_df["cost"]`. Agregue título y etiquetas de los ejes.

In [ ]:
# TODO: complete la visualizacion de la curva de costo
plt.figure()
# TODO: plt.plot(...)
plt.title("Evolución de la función de costo")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.show()

## Ejercicio 8 — Frontera aprendida e interpretación

Imprima los pesos y el sesgo aprendidos, y grafique la frontera de decisión final con `plot_decision_boundary`.

In [ ]:
print("Pesos aprendidos:", network.weights.ravel())
print("Sesgo aprendido:", network.bias)

# TODO: llame a plot_decision_boundary con network.weights.ravel() y network.bias
# (titulo sugerido: "Frontera de decisión aprendida por la red")

**✍️ Tu respuesta:** interprete los pesos aprendidos: si el peso del ingreso es negativo y el de la deuda es positivo, ¿tiene sentido económico? Recuerde: esto es interpretación **predictiva**, no causal. ¿Por qué no podemos decir que "más ingreso causa menor riesgo" solo con este resultado?

*(escriba aquí)*

## Ejercicio 9 — Evaluación (accuracy)

Como este es un problema de clasificación, calculamos la proporción de aciertos.

In [ ]:
# TODO: calcule y_pred con network.predict(X) y la accuracy comparando con y
y_pred = None
accuracy = None

print("Accuracy:", round(accuracy, 4) if accuracy is not None else None)

---
# Parte 3 — Experimentos con la red

## Ejercicio 10 — ¿Qué pasa si la tasa de aprendizaje es muy alta o muy baja?

La tasa de aprendizaje \(\eta\) controla el tamaño de los pasos en la optimización. Piense en una autoridad monetaria que ajusta la tasa de interés: si ajusta muy lento, tarda mucho en corregir; si ajusta demasiado agresivo, puede generar inestabilidad.

**Entrene tres redes** con \(\eta = 0.001\), \(\eta = 0.1\) y \(\eta = 10\) (1000 épocas cada una) y compare sus curvas de costo en una sola gráfica.

In [ ]:
learning_rates = [0.001, 0.1, 10]
lr_histories = {}

for eta in learning_rates:
    # TODO: cree una Network con learning_rate=eta, seed=42, entrenela 1000 epocas (store_every=10)
    # y guarde su historial (pd.DataFrame(temp_net.history)) en lr_histories[eta]
    pass

plt.figure()
for eta, hist in lr_histories.items():
    plt.plot(hist["epoch"], hist["cost"], label=f"eta={eta}")
plt.title("Efecto de la tasa de aprendizaje")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.legend()
plt.show()

**✍️ Tu respuesta:**
1. ¿Qué tasa de aprendizaje converge más lento?
2. ¿Cuál parece más eficiente para este problema?
3. ¿Observa algún comportamiento problemático (oscilaciones, inestabilidad) con \(\eta=10\)? Si no lo observa, ¿por qué cree que la analogía de la "autoridad monetaria demasiado agresiva" no se materializa aquí? (Pista: piense en qué tan simple es este problema — solo 2 parámetros — y en qué tan grande tendría que ser \(\eta\) para desestabilizar un problema tan simple.)
4. ¿Cómo relacionaría esto con ajustes de política económica demasiado agresivos?

*(escriba aquí)*

## Ejercicio 11 — ¿Qué pasa si inicializamos todos los pesos en cero?

En redes con **muchas neuronas ocultas**, inicializar todos los pesos en cero genera un problema de **simetría**: las neuronas empiezan iguales, reciben los mismos gradientes, aprenden lo mismo, y se pierde diversidad de representación.

**Matiz importante:** aquí tenemos una sola neurona sigmoidal. Entrene una red con `zero_init=True` y compare su curva de costo y accuracy final con la red del Ejercicio 6.

In [ ]:
# TODO: cree zero_net = Network(n_inputs=2, learning_rate=0.5, seed=42, zero_init=True)
# entrenela 1000 epocas (store_every=20), grafique su curva de costo,
# imprima sus pesos/sesgo finales y su accuracy
zero_net = None
zero_history = None
zero_acc = None

print("Pesos aprendidos desde cero:", zero_net.weights.ravel() if zero_net is not None else None)
print("Accuracy desde cero:", zero_acc)

**✍️ Tu respuesta:** ¿por qué esta red de una sola neurona sí logra aprender con pesos en cero, mientras que una red con varias neuronas ocultas podría tener problemas? Escriba la respuesta con sus propias palabras.

*(escriba aquí)*

## Ejercicio 12 — El umbral de clasificación

Hasta ahora usamos el umbral por defecto (0.5): si \(\hat{y} \geq 0.5\), clasificamos como alto riesgo. Pero una institución financiera podría preferir ser más conservadora.

**Calcule, para umbral 0.3 y umbral 0.7:**
- el número de **falsos positivos** (el modelo predijo alto riesgo pero en realidad era bajo riesgo);
- el número de **falsos negativos** (el modelo predijo bajo riesgo pero en realidad era alto riesgo);
- el accuracy total.

Use `network.predict(X, threshold=...)`.

In [ ]:
for th in [0.3, 0.5, 0.7]:
    # TODO: calcule pred_th usando network.predict(X, threshold=th)
    pred_th = None

    # TODO: cuente falsos positivos (pred_th==1 y y==0) y falsos negativos (pred_th==0 y y==1)
    # (pista: use mascaras booleanas y np.sum, por ejemplo:
    #  fp = np.sum((pred_th == 1) & (y == 0)) )
    fp = None
    fn = None
    acc_th = None

    print(f"umbral={th}: accuracy={acc_th}  falsos_positivos={fp}  falsos_negativos={fn}")

**✍️ Tu respuesta:**
1. ¿Qué le pasa a los falsos positivos y a los falsos negativos al subir el umbral de 0.3 a 0.7?
2. Una entidad financiera **conservadora** (que prioriza no prestarle a alguien que no va a pagar) — ¿preferiría un umbral más alto o más bajo? ¿Por qué?
3. ¿Qué costo económico distinto tienen un falso positivo (rechazar/encarecer crédito a alguien que sí habría pagado) y un falso negativo (aprobar crédito a alguien que no paga)? ¿Ese costo debería influir en qué umbral se usa?

*(escriba aquí)*

## Ejercicio 13 (cierre) — Síntesis

En **máximo 8 líneas**, explique con sus propias palabras cómo se conectan estos elementos con un modelo de elección binaria en economía:

- pesos;
- sesgo;
- función sigmoide;
- descenso del gradiente.

**✍️ Tu respuesta:**

*(escriba aquí)*

---
# Cierre del taller

## Ideas clave que debería llevarse

1. Un **perceptrón** es una regla de decisión binaria con umbral — cambia bruscamente de 0 a 1.
2. La **sigmoide** suaviza esa decisión y permite calcular derivadas, lo que habilita el aprendizaje gradual.
3. El aprendizaje consiste en **minimizar una función de costo** (aquí, el MSE).
4. El **descenso del gradiente**, construido paso a paso vía la regla de la cadena, actualiza los pesos en la dirección que más reduce el error.
5. La **tasa de aprendizaje** controla la velocidad y estabilidad de ese proceso.
6. Los pesos aprendidos pueden leerse como señales de **importancia predictiva** — nunca como efectos causales sin un diseño de identificación adecuado.
7. El **umbral de decisión** es una palanca de política que la propia institución elige, con costos económicos distintos según el tipo de error.

## Checklist de entrega

- [ ] Ejercicios 1 a 13 completos (código sin `pass` ni `None` pendientes).
- [ ] Todas las preguntas **✍️ Tu respuesta** contestadas con sus propias palabras.
- [ ] El notebook corre de principio a fin sin errores (`Kernel → Restart & Run All`).

Una red neuronal no es magia: es una máquina de optimización que ajusta parámetros para reducir errores. Para un economista, lo importante no es solo programarla, sino entender qué decisión representa, qué información usa, qué error minimiza, qué supuestos implícitos tiene, y si su comportamiento tiene sentido económico.
